# 🧬 Cancer Prediction Using Machine Learning

## Phase 6-8 — Train-Test Split, Model Selection & Training

**Objective:** Split data, compare 8 algorithms, select and train the best model.

---

### 📋 Table of Contents

1. Setup & Load Data
2. Train-Test Split (Stratified)
3. Feature Scaling
4. Model Comparison (8 Algorithms)
5. Cross-Validation
6. Final Model Training
7. Phase Summary

In [ ]:
# Setup
import os, sys, warnings, json, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

%matplotlib inline
sns.set_style('whitegrid')

src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from utils import print_section_header, format_percentage

# Load data
df = pd.read_csv('../data/processed/breast_cancer_cleaned.csv')

# Load selected features (from Phase 5)
features_path = '../data/processed/selected_features.json'
if os.path.exists(features_path):
    with open(features_path, 'r') as f:
        selected_features = json.load(f)
    print(f"✅ Using {len(selected_features)} selected features from Phase 5")
else:
    selected_features = [col for col in df.columns if col != 'diagnosis']
    print(f"⚠️ No feature selection file found, using all {len(selected_features)} features")

X = df[selected_features]
y = df['diagnosis']

print(f"   Features: {X.shape[1]}, Samples: {X.shape[0]}")
print(f"   Target distribution: {y.value_counts().to_dict()}")

---

## 2. ✂️ Train-Test Split

### Why split the data?

We need to evaluate our model on data it has **never seen** during training.
If we train and test on the same data, the model will simply memorize the answers.

### Split strategies:

| Strategy | When to Use | Pros | Cons |
|----------|------------|------|------|
| **80/20 Split** | Default choice, enough data | Simple, fast | Single evaluation |
| **70/30 Split** | Small dataset, want more test data | Better test estimate | Less training data |
| **K-Fold CV** | Robust evaluation needed | Uses all data for testing | Slower |
| **Stratified Split** | Imbalanced classes | Preserves class ratio | Slightly more complex |

### Our choice: **Stratified 80/20 Split**

- **80/20** because 569 samples is moderate — 80% (455) is enough to train, 20% (114) is enough to test
- **Stratified** because we have class imbalance (63% B, 37% M) — we must preserve this ratio

### What is `random_state`?

It's the **seed** for the random number generator. Using `random_state=42`:
- Makes the split **reproducible** — same split every time you run the code
- Essential for **debugging** — you can't debug if results change each run
- Convention: 42 is the traditional seed (from "Hitchhiker's Guide to the Galaxy")

In [ ]:
# =============================================================================
# CELL 2: Stratified Train-Test Split
# =============================================================================
# CRITICAL: We split BEFORE scaling. If we scale first, the scaler
#       would learn the mean/std from ALL data including the test set.
#       This is DATA LEAKAGE — the model indirectly sees test data.
#
#       Correct order: Split → Fit scaler on TRAIN → Transform BOTH
# =============================================================================

print_section_header("Train-Test Split (Stratified 80/20)", "✂️")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,        # 20% for testing
    random_state=42,        # Reproducibility
    stratify=y              # Preserve class proportions
)

print(f"  Training set:  {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"  Test set:      {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\n  Training target distribution:")
print(f"    Benign (0):    {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)")
print(f"    Malignant (1): {(y_train == 1).sum()} ({(y_train == 1).mean()*100:.1f}%)")
print(f"\n  Test target distribution:")
print(f"    Benign (0):    {(y_test == 0).sum()} ({(y_test == 0).mean()*100:.1f}%)")
print(f"    Malignant (1): {(y_test == 1).sum()} ({(y_test == 1).mean()*100:.1f}%)")
print("\n✅ Class proportions preserved in both sets (stratified split)")

---

## 3. 📏 Feature Scaling

### Why scale features?

| Algorithm | Needs Scaling? | Why |
|-----------|---------------|-----|
| Logistic Regression | ✅ Yes | Gradient descent converges faster |
| KNN | ✅ Yes | Distance-based — large features dominate |
| SVM | ✅ Yes | Kernel calculations assume similar scales |
| Decision Tree | ❌ No | Splits on individual features |
| Random Forest | ❌ No | Ensemble of trees |
| Naive Bayes | ❌ No | Probability-based |

We scale anyway because it **doesn't hurt** tree models and **helps** others.

In [ ]:
# =============================================================================
# CELL 3: Feature Scaling (StandardScaler)
# =============================================================================
# CRITICAL: Fit on TRAINING data only, transform BOTH.
#       This prevents data leakage from test set statistics.
# =============================================================================

print_section_header("Feature Scaling (StandardScaler)", "📏")

scaler = StandardScaler()

# Fit on training data ONLY
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# Transform test data using TRAINING statistics
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("  Before scaling:")
print(f"    Train mean: {X_train.mean().mean():.2f}, std: {X_train.std().mean():.2f}")
print(f"    Test mean:  {X_test.mean().mean():.2f}, std: {X_test.std().mean():.2f}")
print("\n  After scaling:")
print(f"    Train mean: {X_train_scaled.mean().mean():.4f} (≈0), std: {X_train_scaled.std().mean():.4f} (≈1)")
print(f"    Test mean:  {X_test_scaled.mean().mean():.4f}, std: {X_test_scaled.std().mean():.4f}")
print("\n✅ Features scaled — fit on train, transformed both")
print("⚠️ Note: Test mean is not exactly 0 — this is CORRECT because we used training statistics")

### 💡 Why test mean ≠ 0?

The test set mean after scaling won't be exactly 0 because we used the **training set's mean and std**.
This is **correct behavior** — it simulates how the model will handle new, unseen data in production.

**🎯 Interview Question:** *Why must you fit the scaler on training data only?*
> If you fit on all data (including test), the scaler learns statistics from the test set. When you then evaluate, the model indirectly has information about the test distribution, inflating metrics. In production, you won't have future data to compute statistics — you must use training statistics only.

---

## 4. 🤖 Model Comparison (8 Algorithms)

We'll train and compare these algorithms:

| # | Algorithm | Type | Strengths | Weaknesses |
|---|-----------|------|-----------|------------|
| 1 | Logistic Regression | Linear | Fast, interpretable, good baseline | Assumes linearity |
| 2 | Decision Tree | Tree | Interpretable, no scaling needed | Overfits easily |
| 3 | Random Forest | Ensemble | Robust, handles outliers | Slow, less interpretable |
| 4 | KNN | Instance | Simple, no training phase | Slow prediction, sensitive to scale |
| 5 | Naive Bayes | Probabilistic | Very fast, handles small data | Assumes feature independence |
| 6 | SVM | Kernel | Great for high-dim data | Slow on large datasets |
| 7 | Gradient Boosting | Ensemble | High accuracy, handles imbalance | Slow, prone to overfit |
| 8 | XGBoost | Ensemble | State-of-the-art, regularized | Complex hyperparameters |

In [ ]:
# =============================================================================
# CELL 4: Define and Train All Models
# =============================================================================
# WHY: Comparing multiple models prevents bias toward any single algorithm.
#       The best model depends on the data — there's no universally best algorithm
#       (this is the "No Free Lunch" theorem).
# =============================================================================

print_section_header("Training 8 Models", "🤖")

# Define models with sensible default hyperparameters
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, C=1.0
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    ),
    'KNN (k=5)': KNeighborsClassifier(
        n_neighbors=5, n_jobs=-1
    ),
    'Naive Bayes': GaussianNB(),
    'SVM (RBF)': SVC(
        kernel='rbf', probability=True, random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=3, random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=3, random_state=42,
        eval_metric='logloss', verbosity=0
    ),
}

# Train each model and collect metrics
results = []

for name, model in models.items():
    start_time = time.time()
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    
    train_time = time.time() - start_time
    
    # Calculate metrics
    result = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_prob) if y_prob is not None else None,
        'Train Time (s)': round(train_time, 4)
    }
    results.append(result)
    
    emoji = '✅' if result['Recall'] >= 0.95 else '⚠️'
    print(f"  {emoji} {name:25s} | Acc: {result['Accuracy']:.4f} | "
          f"Recall: {result['Recall']:.4f} | F1: {result['F1 Score']:.4f} | "
          f"Time: {train_time:.4f}s")

# Create results DataFrame
results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
print("\n" + "=" * 80)
print("📊 MODEL COMPARISON TABLE")
print("=" * 80)
print(results_df.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 5: Model Comparison Visualization
# =============================================================================

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

fig = go.Figure()

colors = ['#388bfd', '#f0883e', '#f85149', '#3fb950']

for i, metric in enumerate(metrics_to_plot):
    fig.add_trace(go.Bar(
        name=metric,
        x=results_df['Model'],
        y=results_df[metric],
        marker_color=colors[i],
        text=[f'{v:.3f}' for v in results_df[metric]],
        textposition='auto'
    ))

fig.update_layout(
    title='🤖 Model Comparison — All Metrics',
    barmode='group',
    height=500,
    template='plotly_dark',
    xaxis_tickangle=-30,
    yaxis_range=[0.8, 1.02],
    legend=dict(x=0.01, y=0.99)
)

fig.show()

try:
    fig.write_image('../images/model_comparison.png', scale=2)
    print("✅ Saved: images/model_comparison.png")
except Exception as e:
    print(f"⚠️ Could not save: {e}")

---

## 5. 🔄 Cross-Validation (Top 3 Models)

A single train/test split can be misleading. **Cross-validation** gives a more robust estimate.

In [ ]:
# =============================================================================
# CELL 6: Stratified K-Fold Cross-Validation
# =============================================================================
# WHY: A single split might give optimistic or pessimistic results depending
#       on which samples land in which set. K-Fold uses ALL data for testing
#       (in different folds), giving a more reliable performance estimate.
#
# We use Stratified K-Fold to preserve class proportions in each fold.
# =============================================================================

print_section_header("5-Fold Stratified Cross-Validation (Top 3 Models)", "🔄")

# Select top 3 models by F1 score
top_3_names = results_df.head(3)['Model'].tolist()
print(f"Top 3 models: {top_3_names}\n")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scale the FULL X for cross-validation (CV handles train/test internally)
# Note: In production, use Pipeline to avoid leakage within folds
from sklearn.pipeline import Pipeline

cv_results = []

for name in top_3_names:
    # Create a pipeline that scales within each fold
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', models[name])
    ])
    
    # Cross-validate with multiple metrics
    scores_acc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
    scores_f1 = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    scores_recall = cross_val_score(pipe, X, y, cv=cv, scoring='recall')
    
    cv_results.append({
        'Model': name,
        'CV Accuracy': f'{scores_acc.mean():.4f} ± {scores_acc.std():.4f}',
        'CV F1': f'{scores_f1.mean():.4f} ± {scores_f1.std():.4f}',
        'CV Recall': f'{scores_recall.mean():.4f} ± {scores_recall.std():.4f}',
        'Acc Mean': scores_acc.mean(),
        'F1 Mean': scores_f1.mean(),
        'Recall Mean': scores_recall.mean(),
        'Std': scores_f1.std()
    })
    
    print(f"  {name}:")
    print(f"    Accuracy: {scores_acc.mean():.4f} ± {scores_acc.std():.4f}")
    print(f"    F1 Score: {scores_f1.mean():.4f} ± {scores_f1.std():.4f}")
    print(f"    Recall:   {scores_recall.mean():.4f} ± {scores_recall.std():.4f}")
    print()

cv_df = pd.DataFrame(cv_results)

# Select best model: highest F1 with low variance
best_model_name = cv_df.sort_values(['F1 Mean', 'Std'], ascending=[False, True]).iloc[0]['Model']
print(f"🏆 Best model (by CV F1): {best_model_name}")

---

## 6. 🏆 Final Model Training

In [ ]:
# =============================================================================
# CELL 7: Train the Best Model
# =============================================================================
# WHY: We retrain the best model on the training set and evaluate
#       on the test set one final time. This is the model we'll save.
# =============================================================================

print_section_header(f"Training Final Model: {best_model_name}", "🏆")

# Get the best model
best_model = models[best_model_name]

# Train on scaled training data
best_model.fit(X_train_scaled, y_train)

# Final predictions
y_pred_final = best_model.predict(X_test_scaled)
y_prob_final = best_model.predict_proba(X_test_scaled)[:, 1]

print(f"\n📊 Final Test Set Performance:")
print(f"   Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}")
print(f"   Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"   Recall:    {recall_score(y_test, y_pred_final):.4f}")
print(f"   F1 Score:  {f1_score(y_test, y_pred_final):.4f}")
print(f"   AUC-ROC:   {roc_auc_score(y_test, y_prob_final):.4f}")

print(f"\n📋 Classification Report:")
print(classification_report(y_test, y_pred_final, 
                           target_names=['Benign (0)', 'Malignant (1)']))

# Save model artifacts for next notebooks
import joblib

# Save model
model_path = '../models/cancer_prediction_model.pkl'
joblib.dump(best_model, model_path)
print(f"💾 Model saved: {model_path}")

# Save scaler
scaler_path = '../models/scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"💾 Scaler saved: {scaler_path}")

# Save test data for evaluation notebook
test_data = X_test.copy()
test_data['diagnosis'] = y_test.values
test_data['predicted'] = y_pred_final
test_data['probability'] = y_prob_final
test_data.to_csv('../data/processed/test_predictions.csv', index=False)
print(f"💾 Test predictions saved: data/processed/test_predictions.csv")

# Save results table
results_df.to_csv('../data/processed/model_comparison_results.csv', index=False)
print(f"💾 Comparison results saved: data/processed/model_comparison_results.csv")

---

## 📋 Phase 6-8 Summary

### ✔ Summary

- **Split:** Stratified 80/20 — class proportions preserved
- **Scaled:** StandardScaler fit on train only (no leakage)
- **Compared:** 8 algorithms — Logistic Regression, Decision Tree, Random Forest, KNN, Naive Bayes, SVM, Gradient Boosting, XGBoost
- **Cross-validated:** Top 3 models with 5-fold stratified CV using Pipeline
- **Selected & saved:** Best model + scaler

### ✔ What You Learned

| Concept | Description |
|---------|-------------|
| Stratified Split | Preserves class ratio in train/test |
| Data Leakage via Scaling | Must fit scaler on train only |
| No Free Lunch Theorem | No universally best algorithm |
| Cross-Validation Pipeline | Prevents leakage within folds |
| random_state | Seed for reproducibility |

### ✔ Files Created

```
📁 models/
├── cancer_prediction_model.pkl
└── scaler.pkl
📁 data/processed/
├── test_predictions.csv
└── model_comparison_results.csv
📁 images/
└── model_comparison.png
```

---

*Phases 6-8 Complete ✅*